# Lesson 6: CNNs, Sequence Models & Transformer Attention Mechanics

**Runtime Tier**: Google Colab Free Tier **T4 GPU** (`Runtime -> Change runtime type -> T4 GPU`)

Run PyTorch `Conv2d`, `LSTM`, and `MultiHeadSelfAttention` blocks on synthetic image and sequence batches.


In [ ]:
import math
import torch
import torch.nn as nn

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Active PyTorch device:', device)


In [ ]:
# 1. Minimal PyTorch CNN for 1x28x28 images
cnn = nn.Sequential(
    nn.Conv2d(1, 16, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2),
    nn.Flatten(),
    nn.Linear(16 * 14 * 14, 10),
).to(device)

images = torch.randn(8, 1, 28, 28, device=device)
print('CNN logits shape:', cnn(images).shape)

# 2. Minimal PyTorch LSTM for sequences
lstm = nn.LSTM(input_size=32, hidden_size=64, batch_first=True).to(device)
seq = torch.randn(8, 12, 32, device=device)
out_seq, (h_n, c_n) = lstm(seq)
print('LSTM output shape:', out_seq.shape, 'Final hidden:', h_n.shape)


In [ ]:
# 3. Multi-Head Causal Self-Attention in PyTorch
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model=64, n_heads=4, causal=True):
        super().__init__()
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        self.causal = causal
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.proj = nn.Linear(d_model, d_model)

    def forward(self, x):
        B, T, C = x.shape
        qkv = self.qkv(x).view(B, T, 3, self.n_heads, self.d_k).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.d_k)
        if self.causal:
            mask = torch.triu(torch.ones(T, T, device=x.device), diagonal=1).bool()
            att = att.masked_fill(mask, float('-inf'))
        att = torch.softmax(att, dim=-1)
        out = (att @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.proj(out), att

mha = MultiHeadSelfAttention(d_model=64, n_heads=4, causal=True).to(device)
tokens = torch.randn(2, 6, 64, device=device)
y, weights = mha(tokens)
print('MHA output shape:', y.shape, 'Attention weights shape:', weights.shape)
